# Few-shot ticket normalizer

This notebook demonstrates few-shot prompting by transforming an unstructured bug report into a normalized support ticket schema.

## 1. Setup

To use the OpenAI API, you need to have an API key.

1. Create or log into your [OpenAI account](https://platform.openai.com/)
2. Go to [API Keys](https://platform.openai.com/api-keys)
3. Click `Create new secret key`, name it, copy it immediately
4. Add billing/payment details so the key becomes active

Once you have the API key, export it as an environment variable called `OPENAI_API_KEY`. If this environment variable is not defined, you will be asked for your API key.

In [ ]:
import json
import os
from textwrap import dedent

from openai import OpenAI

client = OpenAI()
DEFAULT_MODEL = os.getenv("MODEL", "gpt-4o-mini")

In [ ]:
def build_messages(report):
    return [
        {
            "role": "system",
            "content": "You normalize bug reports into compact support tickets. Return only valid JSON with the keys title, category, priority, summary, and next_step.",
        },
        {
            "role": "user",
            "content": "Bug report:\nThe mobile app crashes when I try to upload a 20 MB PDF. It worked yesterday.",
        },
        {
            "role": "assistant",
            "content": json.dumps({
                "title": "Crash when uploading large PDF",
                "category": "file_upload",
                "priority": "high",
                "summary": "The mobile app crashes during PDF upload when the file is around 20 MB.",
                "next_step": "Investigate upload limits and crash logs for the mobile client.",
            }, indent=2),
        },
        {
            "role": "user",
            "content": "Bug report:\nSearch results on the dashboard take a long time to load, especially in the morning.",
        },
        {
            "role": "assistant",
            "content": json.dumps({
                "title": "Slow dashboard search",
                "category": "performance",
                "priority": "medium",
                "summary": "Dashboard search becomes slow, with the issue being most visible during morning usage.",
                "next_step": "Check query latency, indexing, and peak-time load on the search service.",
            }, indent=2),
        },
        {
            "role": "user",
            "content": f"Bug report:\n{report}\n\nReturn only JSON.",
        },
    ]


def normalize_report(report, model=DEFAULT_MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=build_messages(report),
        temperature=0,
        response_format={"type": "json_object"},
    )
    content = response.choices[0].message.content or "{}"
    return json.loads(content)

In [ ]:
report = dedent("""
The app logs me out whenever I close the browser tab.
I already checked the password manager, and I have to sign in every time.
""").strip()

ticket = normalize_report(report)

print("=== Few-shot ticket normalizer ===")
print("Bug report:")
print(report)
print("\nNormalized ticket:")
print(json.dumps(ticket, indent=2))